In [14]:
from urllib.parse import urlsplit, urlunsplit

import pandas as pd
import ast
import re

# Lectura de datos

In [15]:
prompts_revision = pd.read_csv('../data/revision_manual/df_mel.csv').prompt

In [16]:
resultados_gemini = pd.read_csv('../data/results_gemini_links.csv')
resultados_gemini.shape

(2135, 6)

In [17]:
resultados_mistral = pd.read_csv('../data/results_mistral.csv')
resultados_mistral.shape

(2135, 4)

In [18]:
resultados_gpt = pd.read_csv('../data/results_gpt_web_search.csv')
resultados_gpt.shape

(1042, 4)

In [19]:
resultados_gemini = resultados_gemini[resultados_gemini.prompt.isin(prompts_revision)]

In [20]:
resultados_mistral = resultados_mistral[resultados_mistral.prompt.isin(prompts_revision)]

In [21]:
resultados_gpt = resultados_gpt[resultados_gpt.prompt.isin(prompts_revision)]

In [29]:
faltan_gpt = set(prompts_revision) - set(resultados_gpt.prompt)
faltan_gpt

{'I want to write an article about: "Standard corporate reporting can hide the long-term climate cost of building data centers and buying new hardware.". Give me references that support the hypothesis related to that topic.',
 'I\'m analyzing this article https://wjarr.com/sites/default/files/WJARR-2024-1394.pdf, what are other references related to the statement "A layered security posture, provenance controls + decentralized training + hardened deployment + IP protection, matches best practice thinking, but introduces measurable performance/complexity trade-offs.".',
 'I\'m analyzing this article https://www.sciencedirect.com/science/article/pii/S0959652622032115, what are other references related to the statement "“Green” electricity can reduce carbon but still carry high water/land trade-offs Assessing data-storage sustainability using only CO₂ is incomplete: the electricity mix can lower carbon footprints while increasing water and/or land footprints.".',
 'Sources pro or against 

In [30]:
results_gpt2 = pd.read_csv('../data/results_gpt.csv')
results_gpt2 = results_gpt2[results_gpt2.prompt.isin(faltan_gpt)]
results_gpt2.dropna(subset='references', inplace = True)

In [31]:
results_gpt2

,prompt,result,references,tokens
70,"I want to write an article about: ""Standard co...",Here are several strong references supporting ...,['https://www.theguardian.com/technology/2024/...,NaN
719,What is the current consensus on the statement...,Here’s the current consensus on the statement:...,['https://arxiv.org/abs/2506.07712?utm_source=...,NaN


In [32]:
results_gpt3 = pd.read_csv('../data/results_gptv3.csv')
results_gpt3 = results_gpt3[results_gpt3.prompt.isin(faltan_gpt)]
results_gpt3

,prompt,result,references,tokens
439,"Sources pro or against the statement ""Data poi...",Data poisoning and adversarial inputs are reco...,['https://www.ebryx.com/blogs/what-is-data-poi...,NaN
616,I'm analyzing this article https://www.science...,"The statement highlights that while ""green"" el...",['https://arxiv.org/abs/2512.08725?utm_source=...,NaN
650,I'm analyzing this article https://wjarr.com/s...,The statement in the article highlights the tr...,['https://www.defendify.com/wp-content/uploads...,NaN


# Revisión de resultados

In [20]:
for i, prompt in resultados_gemini.iterrows():
    print('\n\n\n---------------------> PROMPT', prompt.prompt, '\n')
    print('Respuesta:', prompt.result, '\n\n')
    for link in ast.literal_eval(prompt.references_real):
        print('LINK: ', link)




---------------------> PROMPT I want to write an article about: "Standard corporate reporting can hide the long-term climate cost of building data centers and buying new hardware.". Give me references that support the hypothesis related to that topic. 

Respuesta: Standard corporate reporting often fails to fully capture the extensive, long-term climate costs associated with building data centers and acquiring new hardware, leading to an underestimation of the true environmental impact. This 


LINK:  Error: MissingSchema: Invalid URL '': No scheme supplied. Perhaps you meant https://?



---------------------> PROMPT Explain the following statement: "Workers may overestimate the true benefits they get from AI chatbots. Self-reports of large gains can exceed what is reflected in administrative outcomes, suggesting a risk of inflated perceptions.". Include references for each key claim. 

Respuesta: The statement "Workers may overestimate the true benefits they get from AI chatbots. 

In [32]:
import textwrap

for i, prompt in resultados_mistral.iterrows():
    print('\n\n\n---------------------> PROMPT', prompt.prompt, '\n')
    print(textwrap.fill(prompt.result, width=100))

    if len(ast.literal_eval(prompt.references)) == 0:
        print('\n------SIN REFERENCIAS')
    for link in ast.literal_eval(prompt.references):
       print('REFERNCIA: ', link)




---------------------> PROMPT I want to write an article about: "Standard corporate reporting can hide the long-term climate cost of building data centers and buying new hardware.". Give me references that support the hypothesis related to that topic. 

Title: The Carbon Footprint of Data Centers and IT Hardware: A Long-term Perspective  1. Strand, J.
(2019). The carbon footprint of data centers: A review of the literature. Journal of Cleaner
Production, 223, 119-130.    - Provides an overview of the carbon footprint of data centers,
including the energy consumption and greenhouse gas emissions associated with their operation.  2.
Bock, L., & Bock, M. (2018). The carbon footprint of IT hardware: A life cycle assessment. Journal
of Cleaner Production, 191, 118-128.    - Examines the carbon footprint of IT hardware throughout
its lifecycle, from manufacturing to disposal, and highlights the importance of considering these
emissions when assessing the long-term climate cost of building

In [40]:
resultados_gpt

,prompt,result,references,tokens
22,"Explain the following statement: ""Workers may ...",The statement suggests that workers might over...,['https://bfi.uchicago.edu/wp-content/uploads/...,NaN
88,"Explain the following statement: ""Adolescents ...",Adolescents experiencing psychological depende...,['https://www.thejustice.org/article/2025/11/t...,NaN
89,"Explain the following statement: ""Adolescents ...",Adolescents may experience distress when their...,['https://www.thejustice.org/article/2025/11/t...,NaN
599,Evaluate the strength of evidence across the f...,Several studies have examined the impact of AI...,['https://bfi.uchicago.edu/wp-content/uploads/...,NaN
949,Evaluating the body of evidence on the stateme...,"The statement that ""The widespread disseminati...",['https://www.mdpi.com/2076-0760/13/8/418?utm_...,NaN


In [51]:
import textwrap
df = pd.DataFrame(columns = ['prompt', 'reference'])
for i, prompt in resultados_gpt.iterrows():
    print('\n\n\n---------------------> PROMPT', prompt.prompt, '\n')
    print(textwrap.fill(prompt.result, width=100))

    if len(ast.literal_eval(prompt.references)) == 0:
        print('\n------SIN REFERENCIAS')
    for link in ast.literal_eval(prompt.references):
       print('REFERNCIA: ', link)
       df.loc[len(df)] = {
           'prompt': prompt.prompt,
           'reference': link
       }





---------------------> PROMPT Explain the following statement: "Workers may overestimate the true benefits they get from AI chatbots. Self-reports of large gains can exceed what is reflected in administrative outcomes, suggesting a risk of inflated perceptions.". Include references for each key claim. 

The statement suggests that workers might overestimate the benefits they derive from AI chatbots,
with self-reported gains often exceeding actual outcomes. This discrepancy indicates a potential
risk of inflated perceptions regarding AI's impact on productivity and well-being.  **Overestimation
of Benefits:**  Studies have found that while workers report time savings and increased productivity
from using AI chatbots, these self-reported benefits do not translate into significant improvements
in earnings or labor market outcomes. For instance, a study by the Becker Friedman Institute found
that despite workers reporting time savings of approximately 7%, these did not correspond with
h

In [33]:
results_gpt2

,prompt,result,references,tokens
70,"I want to write an article about: ""Standard co...",Here are several strong references supporting ...,['https://www.theguardian.com/technology/2024/...,NaN
719,What is the current consensus on the statement...,Here’s the current consensus on the statement:...,['https://arxiv.org/abs/2506.07712?utm_source=...,NaN


In [34]:
import textwrap
df = pd.DataFrame(columns = ['prompt', 'reference'])
for i, prompt in results_gpt2.iterrows():
    print('\n\n\n---------------------> PROMPT', prompt.prompt, '\n')
    print(textwrap.fill(prompt.result, width=100))

    if len(ast.literal_eval(prompt.references)) == 0:
        print('\n------SIN REFERENCIAS')
    for link in ast.literal_eval(prompt.references):
       print('REFERNCIA: ', link)
       df.loc[len(df)] = {
           'prompt': prompt.prompt,
           'reference': link
       }





---------------------> PROMPT I want to write an article about: "Standard corporate reporting can hide the long-term climate cost of building data centers and buying new hardware.". Give me references that support the hypothesis related to that topic. 

Here are several strong references supporting the hypothesis that standard corporate reporting often
obscures the long-term climate costs of building data centers and purchasing new hardware:  • A
Guardian analysis found that emissions from in-house data centers of major tech firms—Google,
Microsoft, Meta, and Apple—are likely about 662% higher than officially reported. This discrepancy
stems from undercounted Scope 3 emissions, including those from construction and hardware
manufacturing, which are often omitted or misclassified in corporate disclosures.
([theguardian.com](https://www.theguardian.com/technology/2024/sep/15/data-center-gas-emissions-
tech?utm_source=openai))  • A Uptime Institute report reveals that most data center 

In [35]:
df = pd.DataFrame(columns = ['prompt', 'reference'])
for i, prompt in results_gpt3.iterrows():
    print('\n\n\n---------------------> PROMPT', prompt.prompt, '\n')
    print(textwrap.fill(prompt.result, width=100))

    if len(ast.literal_eval(prompt.references)) == 0:
        print('\n------SIN REFERENCIAS')
    for link in ast.literal_eval(prompt.references):
       print('REFERNCIA: ', link)
       df.loc[len(df)] = {
           'prompt': prompt.prompt,
           'reference': link
       }





---------------------> PROMPT Sources pro or against the statement "Data poisoning and adversarial inputs are core threat classes that can systematically distort model behavior (not just cause random errors).". 

Data poisoning and adversarial inputs are recognized as significant threat classes that can
systematically distort model behavior. Data poisoning involves injecting malicious data into the
training set, leading to misclassifications, degraded accuracy, and biased outputs. For instance, in
healthcare applications, poisoned models can misdiagnose conditions or suppress early-stage cancer
detection. ([ebryx.com](https://www.ebryx.com/blogs/what-is-data-poisoning-in-ai-how-it-works-and-
how-to-prevent-it?utm_source=openai))  Adversarial inputs, on the other hand, are carefully crafted
perturbations designed to deceive models into making incorrect predictions. These inputs can cause
models to misclassify benign data as malicious or vice versa, undermining the model's reliability

In [60]:
import openpyxl
from openpyxl.utils.dataframe import dataframe_to_rows

# 1. Crear un libro de trabajo nuevo
wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Sheet1"

# 2. Convertir el DataFrame y escribirlo en la hoja
for r in dataframe_to_rows(df, index=False, header=True):
    ws.append(r)

# 3. (Opcional) Añadir el autofiltro correctamente
# Esto aplica el filtro a todas las columnas basándose en el tamaño del DF
ws.auto_filter.ref = ws.dimensions

# 4. Guardar
wb.save('borrar.xlsx')
print("¡Archivo guardado con éxito usando openpyxl directo!")

¡Archivo guardado con éxito usando openpyxl directo!


# GPT - revisión 

In [12]:
resultados_gpt = pd.read_csv('../data/results_gpt_web_search.csv')
results_gpt2 = pd.read_csv('../data/results_gpt.csv')
prompts = pd.read_csv('../data/prompts.csv')

In [13]:
prompts.shape, resultados_gpt.shape, results_gpt2.shape

((2135, 1), (1042, 4), (2135, 4))

In [20]:
prompts_revision_na = results_gpt2[results_gpt2.references.isna()]
len(prompts_revision_na)

1398

In [25]:
prompt_faltan = prompts_revision_na[~prompts_revision_na.prompt.isin(resultados_gpt.prompt)]
prompt_faltan = prompt_faltan[['prompt']]
prompt_faltan.to_csv('../data/prompts_correr_gpt.csv', index = None)